In [1]:
import os
os.environ['HF_HOME'] = '/sci/archive/michall/roeizucker/huggingface_modles_cache' 
!export HF_HOME="/sci/archive/michall/roeizucker/huggingface_modles_cache"
!set HF_HOME="/sci/archive/michall/roeizucker/huggingface_modles_cache"

In [2]:
# from huggingface_hub import login
# login()


In [ ]:
# from transformers import AutoTokenizer, AutoModelForMaskedLM

# model_name = "InstaDeepAI/NTv3_650M_pre"

# # Load model and tokenizer
# tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True)



In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "InstaDeepAI/NTv3_8M_pre"

# Load model and tokenizer
tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)



In [4]:
# # Tokenize input sequences
# batch = tok(["ATCGNATCG", "ACGT"], add_special_tokens=False, padding=True, pad_to_multiple_of=128, return_tensors="pt")

# # Run model
# out = model(**batch)

# # Print output shapes
# print(out.logits.shape)       # (B, L, V = 11)


In [5]:
# from transformers import AutoConfig
# cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
# print(getattr(cfg, "auto_map", None))


In [6]:
# from datasets import Dataset
# # src/data_extractor.py
# import sys
# import os, shutil
# sys.path.insert(0, os.path.abspath("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/refactored_code/src"))
# import datasets
# datasets.disable_caching()
# import pandas as pd
# from typing import Dict, Tuple
# # from Bio import SeqIO
# from random import randint
# # from sklearn.model_selection import train_test_split
# from utils.formatting import (
#     combine_rows_to_multiple_instances_format,
#     add_borders_to_multiple_instance_format_df,
#     create_labels_formultiple_instance_format_df,
#     add_sequence_formultiple_instance_format_df,
#     combine_cpg_dfs,
#     create_methyl_variability_df,
#     create_length_window_id,
#     group_methyl_variability_df,
#     WINDOW_ID_COLUMN_NAME,
#     MAX_DIFF_VARIABILITY_TYPE,  #TODO: these values might need to be defined in a constants file
#     STD_VARIABILITY_TYPE,
#     QUANTILE_SEPERATION_TYPE,
# )
# from utils.bigwig_utils import get_preprocessed_encode_cpg_dataframe
# from datasets import Dataset, DatasetDict, load_from_disk
# from utils.dataset_utils import dataset_generator_wrapper
# dataset = Dataset.from_generator(dataset_generator_wrapper(
#     "/sci/archive/michall/roeizucker/huggingface_datasets_dir/temp_test2025-12-1413:25:55.85asdf.csv", 10,1))

In [7]:
# encoded_dataset = dataset.map(lambda examples: tok(examples['seq']))

In [8]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

repo = "InstaDeepAI/NTv3_650M_pre"

tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
backbone = AutoModelForMaskedLM.from_pretrained(repo, trust_remote_code=True)

batch = tok(
    ["ATCGNATCG", "ACGT"],
    add_special_tokens=False,
    padding=True,
    pad_to_multiple_of=128,
    return_tensors="pt",
)
out = backbone(**batch, output_hidden_states=True, return_dict=True)

# Final embedding is hidden_states[-1], shape (B, L, 1536) for this checkpoint. :contentReference[oaicite:2]{index=2}
print(out.hidden_states[-1].shape)


torch.Size([2, 128, 1536])


In [13]:
import torch
import torch.nn as nn
from transformers.modeling_outputs import TokenClassifierOutput

class NTv3ForTokenClassification(nn.Module):
    def __init__(self, ntv3_mlm, num_labels: int, hidden_size: int = 1536, dropout: float = 0.1):
        super().__init__()
        self.ntv3_mlm = ntv3_mlm
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.ntv3_mlm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
            **kwargs,
        )

        # Use final embedding (after deconv tower), per NTv3 model card. :contentReference[oaicite:4]{index=4}
        x = out.hidden_states[-1]  # (B, L, 1536)
        logits = self.classifier(self.dropout(x))  # (B, L, num_labels)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

        return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=out.hidden_states)

num_labels = 8  # set yours
model = NTv3ForTokenClassification(backbone, num_labels=num_labels)


In [10]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tok,
    padding=True,
    pad_to_multiple_of=128,
    label_pad_token_id=-100,
)


In [12]:
# from transformers import AutoConfig, AutoModelForMaskedLM, AutoTokenizer

# repo = "InstaDeepAI/NTv3_650M_pre"  # or your checkpoint repo
# cfg = AutoConfig.from_pretrained(repo, trust_remote_code=True)
# tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
# mlm = AutoModelForMaskedLM.from_pretrained(repo, trust_remote_code=True)
# model = NTv3ForTokenClassification(mlm, num_labels=1)


In [ ]:
# model = NTv3ForTokenClassification(backbone, num_labels=num_labels)


In [14]:
import sys
import os
sys.path.insert(0, os.path.abspath("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/refactored_code"))
from src.utils.trainer_utils import TokenRegressionTrainer, get_training_args
from transformers import Trainer, TrainingArguments, TrainerCallback
from datasets import load_from_disk


In [15]:
dataset = load_from_disk("/sci/archive/michall/roeizucker/huggingface_datasets_dir/ntv3-train")

In [16]:
# args = get_training_args(1e-6,2,2,4,2,False,"/sci/labs/michall/roeizucker/trained_huggingface_models_location/ntv3_test",None,False,"epoch",10000,2)
args = TrainingArguments(
output_dir= "/sci/labs/michall/roeizucker/trained_huggingface_models_location/ntv3_test",
learning_rate=1e-6,            # 1e-5 if full fine-tune
num_train_epochs=3,
save_strategy="steps",
load_best_model_at_end=False,
metric_for_best_model=None,
report_to="none"
)


In [17]:

from src.utils.metrics_utils import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    mcc_score,
    compute_metrics,
    compute_metrics_regression
)

In [20]:
trainer = TokenRegressionTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    tokenizer=tok,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_153535/1357330576.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `TokenRegressionTrainer.__init__`. Use `processing_class` instead.
  trainer = TokenRegressionTrainer(


In [21]:
trainer.train()

RuntimeError: The size of tensor a (336) must match the size of tensor b (337) at non-singleton dimension 2